In [12]:
import pandas as pd

courses = pd.read_csv('../data/Coursera_courses.csv')
reviews = pd.read_csv('../data/Coursera_reviews.csv')

df = reviews.merge(courses, on='course_id', how='inner')

print("Shape:", df.shape)
print("\nMissing values:\n", df.isna().sum())
print("\nRating distribution:\n", df['rating'].value_counts().sort_index())
print("\nUnique courses:", df['course_id'].nunique())
print("Unique institutions:", df['institution'].nunique())
print("\nDuplicate reviews:", df.duplicated(subset=['reviews', 'reviewers', 'course_id']).sum())

Shape: (1454711, 8)

Missing values:
 reviews         153
reviewers         0
date_reviews      0
rating            0
course_id         0
name              0
institution       0
course_url        0
dtype: int64

Rating distribution:
 rating
1      17354
2      16188
3      48303
4     226702
5    1146164
Name: count, dtype: int64

Unique courses: 604
Unique institutions: 132

Duplicate reviews: 934783


## 1. Data Quality Check
The merged dataset has 1.45M reviews, but an initial check shows over 900k rows share the same review text, reviewer and course. Before removing them, we check whether these are true duplicates (scraped twice) or different reviewers with the same abbreviated name.

In [13]:
# duplicate checks
loose_dupes  = df.duplicated(subset=['reviews', 'reviewers', 'course_id']).sum()
strict_dupes = df.duplicated(subset=['reviews', 'reviewers', 'date_reviews', 'course_id']).sum()
full_dupes   = df.duplicated().sum()

print(f"Duplicates (text + reviewer + course): {loose_dupes:,}")
print(f"Duplicates (+ same date):              {strict_dupes:,}")
print(f"Full-row duplicates:                   {full_dupes:,} ({full_dupes / len(df):.1%} of the dataset)")

# how many times does each review repeat?
counts = df.groupby(['reviews', 'reviewers', 'date_reviews', 'course_id']).size()
print("\nMost repeated review appears", counts.max(), "times")
print("\nHow often reviews repeat:\n", counts.value_counts().sort_index())

Duplicates (text + reviewer + course): 934,783
Duplicates (+ same date):              934,764
Full-row duplicates:                   934,764 (64.3% of the dataset)

Most repeated review appears 5 times

How often reviews repeat:
 1     51247
2      2704
3    465862
4        48
5        25
Name: count, dtype: int64


### Finding: 64% of rows were duplicates
934,764 rows are exact copies, and ~466k reviews appear exactly three times. This pattern suggests the duplication comes from the source files rather than genuine repeat reviews, so we trace the cause before removing them.

In [14]:
# is the duplication in the source files or caused by the merge?
print("Duplicate rows in reviews file:", reviews.duplicated().sum())
print("Duplicate course_ids in courses file:", courses['course_id'].duplicated().sum())

# courses are listed more than once?
dup_courses = courses[courses['course_id'].duplicated(keep=False)]
print("\nCourses listed more than once:", dup_courses['course_id'].nunique())
print(dup_courses.sort_values('course_id').head(6))

Duplicate rows in reviews file: 934764
Duplicate course_ids in courses file: 0

Courses listed more than once: 0
Empty DataFrame
Columns: [name, institution, course_url, course_id]
Index: []


In [15]:
# drop exact duplicates and missing reviews
df_clean = df.drop_duplicates().dropna(subset=['reviews']).reset_index(drop=True)

# map ratings to sentiment
df_clean['sentiment'] = df_clean['rating'].map(
    {1: 'Negative', 2: 'Negative', 3: 'Neutral', 4: 'Positive', 5: 'Positive'}
)

print("Before:", len(df))
print("After: ", len(df_clean))
print("\nSentiment distribution:")
print(df_clean['sentiment'].value_counts())
print(df_clean['sentiment'].value_counts(normalize=True).round(3))

Before: 1454711
After:  519886

Sentiment distribution:
sentiment
Positive    489466
Neutral      17728
Negative     12692
Name: count, dtype: int64
sentiment
Positive    0.941
Neutral     0.034
Negative    0.024
Name: proportion, dtype: float64


Class balance is almost unchanged (94.1% Positive, 3.4% Neutral, 2.4% Negative), so accuracy alone would be misleading. We use **macro F1** as the main metric.